笔记内容参考博客:
- [狗都能看懂的DDPM论文详解 CSDN原文链接](https://blog.csdn.net/weixin_42392454/article/details/137458318)
- [扩散模型之DDPM 知乎内容](https://zhuanlan.zhihu.com/p/563661713)


# 总结


- 简单来说，扩散模型包含两个过程：
    - 前向扩散过程和反向生成过程，前向扩散过程是对一张图像逐渐添加高斯噪音直至变成随机噪音，
    - 而反向生成过程是去噪音过程

扩散模型与其它主流生成模型的对比如下所示：
<div style="background-color:white; padding:10px; border-radius:5px; width: 50%; margin: auto;">
    <image src="./assets/ddpm_vs_gan_vs_vae.png" alt="Diffusion vs GAN vs VAE" style="max-width:100%;">
    <p style="text-align:center; font-size:12px; color:#555;">图1: 扩散模型与GAN和VAE的对比</p>
</div>

## 扩散模型原理

扩散模型包括两个过程：前向过程（forward process）和反向过程（reverse process），其中前向过程又称为扩散过程（diffusion process），如下图所示。无论是前向过程还是反向过程都是一个参数化的马尔可夫链（Markov chain），其中反向过程可以用来生成数据，这里我们将通过变分推断来进行建模和求解。

<div style="background-color:white; padding:10px; border-radius:5px; width: 70%; margin: auto;">
    <image src="./assets/pgm_diagram_xarrow_small.png" alt="Forward and Reverse Process" style="max-width:100%;">
    <p style="text-align:center; font-size:12px; color:#555;">图2: 扩散模型的前向过程和反向过程</p>
</div>

### 扩散过程

扩散过程是指对数据逐渐增加高斯噪音直至数据变成随机噪音的过程。对于原始数据$x_0 \sim q(x_0)$，总共包含$T$步的扩散过程的每一步都是对上一步得到的数据按如下方式增加高斯噪音：
$$
q(x_t|x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} x_{t-1}, \beta_t I)
$$

- $\{\beta_t\}_{t=1}^T$(0-1间)为每一步所采用的方差。$\beta_t$往往是逐渐增大的
- 整个扩散过程是一个马尔可夫链，可以表示为：
$$
q(x_{1:T}|x_0) = \prod_{t=1}^T q(x_t|x_{t-1})
$$

<!-- webp -->
<div style="background-color:white; padding:10px; border-radius:5px; width: 70%; margin: auto;">
    <image src="./assets/forward_diffusion_process.webp">
    <p style="text-align:center; font-size:12px; color:#555;">图3: 扩散过程示意图</p>
</div>

- 扩散过程是固定的，例如DDPM使用一个线性的variance schedule。那么我们可以直接从任意一步$t$的$x_t$通过如下公式采样：
$$\begin{aligned}
x_t &= \sqrt{\alpha_t} x_{t-1} + \sqrt{1 - \alpha_t} \epsilon_{t-1}, & \epsilon_{t-1} \sim \mathcal{N}(0, I) \\
&= \sqrt{\alpha_t} \left( \sqrt{\alpha_{t-1}}x_{t-2} + \sqrt{1-\alpha_{t-1} } \epsilon_{t-2} \right) + \sqrt{1 - \alpha_t} \epsilon_{t-1} \\
&= \sqrt{\alpha_t \alpha_{t-1}} x_{t-2} + \sqrt{ \sqrt{\alpha_t - \alpha_t \alpha_{t-1}}^2 + \sqrt{1 - \alpha_t}^2 } \bar{\epsilon}_{t-2} & \bar{\epsilon}_{t-2} \ \text{merges two Gaussians} (*). \sim \mathcal{N}(0, I)\\
&= \sqrt{\alpha_t \alpha_{t-1}} x_{t-2} + \sqrt{1 - \alpha_t \alpha_{t-1}} \bar{\epsilon}_{t-2} \\
&= \ldots \\
&= \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, & \epsilon \sim \mathcal{N}(0, I) \\
\end{aligned}$$

- $\alpha_t = 1 - \beta_t$
- $\bar{\epsilon}_{t-2}$是两个高斯噪音$\epsilon_{t-1}$和$\epsilon_{t-2}$的合并，仍然服从标准正态分布:
    - $\alpha \sim \mathcal{N}(0, \sigma_1^2 I)$, $\beta \sim \mathcal{N}(0, \sigma_2^2I)$
    - $k_1 \alpha + k_2 \beta \sim \mathcal{N}(0, (k_1^2 \sigma_1^2 + k_2^2 \sigma_2^2) I)$
    - $\sqrt{\alpha_t (1 - \alpha_{t-1})}\epsilon_{t-2} + \sqrt{1 - \alpha_t}\epsilon_{t-1} = \sqrt{\alpha_t (1 - \alpha_{t-1}) + (1 - \alpha_t) } \bar\epsilon_{t-2}$
- 上述过程将两个方差不同的高斯分布$\mathcal{N}(0, \sigma_1^2 I)$和$\mathcal{N}(0, \sigma_2^2 I)$合并成一个新的高斯分布$\mathcal{N}(0, (\sigma_1^2 + \sigma_2^2) I)$，反重参数化后，我们得到:
$$\begin{aligned}
q(x_t|x_0) &= \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}x_0, (1 - \bar{\alpha}_t) I) \\
\end{aligned}$$
- 扩散过程的这个特性很重要。首先，我们可以看到$x_t$其实可以看成是原始数据$x_0$和随机噪音$\epsilon$的线性组合，其中$\sqrt{\bar{\alpha}}_t$和$\sqrt{1 -\bar\alpha_t}$为组合系数，它们的平方和等于1，我们也可以称两者分别为signal_rate和noise_rate（见[Denoising Diffusion Implicit Models](https://keras.io/examples/generative/ddim/#diffusion-schedule)和[Improved Denoising Diffusion Probabilistic Models](https://arxiv.org/abs/2102.09672)所设计的cosine schedule），因为这样处理更直接，比如我们直接将$\bar{\alpha}_T$设定为一个接近0的值，那么就可以保证最终得到的$x_T$近似为一个随机噪音。其次，后面的建模和分析过程将使用这个特性。


<div style="background-color:white; padding:10px; border-radius:5px; width: 70%; margin: auto;">
    <image src="./assets/diffusion_kernel.webp">
    <p style="text-align:center; font-size:12px; color:#555;">图4: 扩散核心</p>
</div>

#### 反向过程

扩散过程是将数据噪音化，那么反向过程就是一个去噪的过程，如果我们知道反向过程的每一步的真实分布$q(x_{t-1}|x_t)$，那么从一个随机噪音$x_T \sim \mathcal{N}(0, I)$开始，逐渐去噪就能生成一个真实的样本，所以反向过程也就是生成数据的过程。
<div style="background-color:white; padding:10px; border-radius:5px; width: 70%; margin: auto;">
    <image src="./assets/reverse_diffusion_process.webp">
    <p style="text-align:center; font-size:12px; color:#555;">图5: 反向过程示意图</p>
</div>

估计分布$q(x_{t-1}|x_t)$需要用到整个训练样本，我们可以用神经网络来估计这些分布。这里，我们将反向过程也定义为一个马尔卡夫链，只不过它是由一系列用神经网络参数化的高斯分布来组成：

$$\begin{aligned}
p_\theta(x_{0:T}) &= p(x_T) \prod_{t=1}^T p_\theta(x_{t-1}|x_t) \qquad p_\theta(x_{t-1}|x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t)) \\

\end{aligned}$$